# Wearable AI for Diabetes — D1NAMO Phase 1: Benchmark Glucose Forecasting

**B.Tech Final Year Project | Biomedical AI & Wearable Sensing**

### Project Overview & Objective
The objective of **Phase 1** is to develop, evaluate, and benchmark machine learning and deep learning models for predicting blood glucose levels **30 minutes and 60 minutes ahead** in Type-1 diabetic individuals under free-living conditions.

### Multimodal Inputs
- **Glucose History (Sliding Window):** Past lags ($t-5, t-10, t-15, t-20, t-25, t-30$ min), 30-min rolling mean/std, and 15-min rate of change.
- **Cardiovascular Telemetry:** Heart Rate (HR), and raw 250 Hz ECG-derived Heart Rate Variability metrics (**SDNN**, **RMSSD**, **pNN50**).
- **Physical Activity & Accelerometry:** Movement magnitude from the Zephyr BioHarness chest strap.
- **Thermal & Respiratory Signals:** Wearable Device Temperature (`DeviceTemp`) and Breathing Rate (`BR`).
- **Circadian & Rest Encodings:** Sinusoidal time-of-day harmonics (SweetDeep methodology) and overnight rest indicators.

### Rigorous Validation Scheme
In accordance with clinical standards, models are evaluated using **Leave-One-Subject-Out (LOSO) Cross-Validation** across all 9 subjects ($N=9$ folds). In each fold, the model trains on 8 subjects and evaluates on the single held-out subject to prevent intra-patient data leakage.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

from clarke_error_grid import evaluate_clarke_grid, plot_clarke_error_grid

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print('Libraries loaded successfully! PyTorch CUDA Available:', torch.cuda.is_available())

## Step 1 — Load Synchronized Multimodal Dataset
We load the unified 5-minute synchronized multimodal dataset containing continuous CGM glucose readings, raw ECG-extracted HRV parameters, BioHarness telemetry, and circadian features across all 9 subjects.

In [ ]:
data_path = 'data/d1namo_multimodal_master.parquet'
if not os.path.exists(data_path):
    data_path = 'data/d1namo_multimodal_master.csv'
    df = pd.read_csv(data_path)
else:
    df = pd.read_parquet(data_path)

print(f'Total Dataset Rows: {len(df)}')
print(f'Subjects: {df["subject_id"].nunique()} ({sorted(df["subject_id"].unique())})')
display(df.head(5))

## Step 2 — Exploratory Data Analysis & Physiological Signal Distributions
We inspect the glycemic distributions across subjects, Time-in-Range (TIR) metrics, and correlation with wearable sensors.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Glucose distribution by subject
sns.boxplot(data=df.dropna(subset=['glucose_mgdl']), x='subject_id', y='glucose_mgdl', ax=axes[0], palette='Blues_r')
axes[0].axhline(70, color='red', linestyle='--', label='Hypoglycemia (70 mg/dL)')
axes[0].axhline(180, color='orange', linestyle='--', label='Hyperglycemia (180 mg/dL)')
axes[0].set_title('CGM Glucose Distribution Across Subjects', fontweight='bold')
axes[0].set_xlabel('Subject ID')
axes[0].set_ylabel('Glucose (mg/dL)')
axes[0].tick_params(axis='x', rotation=30)
axes[0].legend(loc='upper right')

# Circadian Glucose Pattern
hour_grp = df.groupby(df['Time'].dt.hour)['glucose_mgdl'].agg(['mean', 'std']).reset_index()
axes[1].plot(hour_grp['Time'], hour_grp['mean'], color='#1f77b4', linewidth=2.5, marker='o', label='Mean Glucose')
axes[1].fill_between(hour_grp['Time'], hour_grp['mean'] - hour_grp['std'], hour_grp['mean'] + hour_grp['std'], color='#1f77b4', alpha=0.2)
axes[1].set_title('Diurnal Circadian Trajectory (Hour of Day)', fontweight='bold')
axes[1].set_xlabel('Hour of Day (0-23)')
axes[1].set_ylabel('Glucose (mg/dL)')
axes[1].set_xticks(range(0, 24, 2))

plt.tight_layout()
plt.show()

## Step 3 — Feature Engineering & Input Modality Sets
We construct two comparative feature sets:
1. **Glucose-Only Baseline:** 6 glucose lags (5 to 30 min), 30-min rolling mean/std, 15-min rate of change, and harmonic circadian encodings (`hour_sin`, `hour_cos`).
2. **Multimodal Feature Set:** Glucose features + Heart Rate (`HR_unified`), ECG-derived **SDNN**, **RMSSD**, Device Temperature (`DeviceTemp`), Accelerometer Activity, and Breathing Rate (`BR`).

In [ ]:
glucose_lags = [f'glucose_lag_{i}min' for i in [30, 25, 20, 15, 10, 5]]
glucose_stats = ['glucose_roll_mean_30min', 'glucose_roll_std_30min', 'glucose_roc_15min']
circadian_features = ['hour_sin', 'hour_cos', 'sin_2phi', 'cos_2phi', 'is_night']
glucose_only_features = glucose_lags + glucose_stats + circadian_features

wearable_features = [c for c in ['HR_unified', 'ECG_SDNN', 'ECG_RMSSD', 'DeviceTemp', 'Activity', 'BR',
                                'HR_unified_roll15', 'ECG_SDNN_roll15', 'Activity_roll15', 'DeviceTemp_roll15']
                     if c in df.columns]

multimodal_features = glucose_only_features + wearable_features

print(f'Glucose-Only Features ({len(glucose_only_features)}):', glucose_only_features)
print(f'Multimodal Features ({len(multimodal_features)}):', multimodal_features)

## Step 4 — Model Architectures & Leave-One-Subject-Out (LOSO) Execution
We evaluate 10 model architectures (Naive Persistence, Linear Regression, Ridge, Random Forest, XGBoost, LightGBM, LSTM, GRU, Temporal CNN, Transformer) across both 30-minute and 60-minute prediction horizons.

In [ ]:
from phase1_benchmark_loso import run_phase1_loso_benchmark
print('Executing Full Phase 1 LOSO Benchmark across all 9 subjects...')
run_phase1_loso_benchmark()

## Step 5 — Benchmark Results & Comparative Analysis
We inspect the aggregated LOSO metrics (MAE, RMSE, Clarke Error Grid Clinical Accuracy Zones A+B).

In [ ]:
results_df = pd.read_csv('results/phase1_loso_results.csv')
display(results_df.sort_values(['Horizon', 'MAE']))

## Step 6 — Clinical Visualizations: MAE Comparisons & Clarke Error Grids

In [ ]:
from IPython.display import Image, display
display(Image('figures/phase1_mae_rmse_comparison.png'))
display(Image('figures/phase1_clarke_grid_best_models.png'))
display(Image('figures/phase1_multimodal_vs_glucose_ablation.png'))

## Step 7 — Key Insights & Clinical Takeaways
1. **Simplicity & Momentum Dominate at Short Horizons:** Linear Regression and Ridge achieve the lowest MAE (15.25 mg/dL at 30 min; 29.64 mg/dL at 60 min) and highest clinical accuracy (98.5% in Zones A+B at 30 min; 93.8% at 60 min), because glucose dynamics over 30–60 min are dominated by near-linear momentum.
2. **Deep Learning Parameterization:** On this multi-subject sample (N=8,374 across 9 subjects), high-capacity sequence models (LSTM, GRU) suffer from cross-subject distributional shift without subject-specific calibration. Regularized linear regressors and gradient-boosted trees generalize superiorly.
3. **Zero Dangerous Zone E Errors:** All top models achieved 0.0% Zone E errors at 30 min and <0.3% at 60 min, ensuring no lethal opposite-treatment recommendations would occur.